# Periodic One-Dimensional Spline Objects
The class ``PeriodicSpline1D`` of the ``splinekit`` library handles continuously defined real functions $f:{\mathbb{R}}\rightarrow{\mathbb{R}},x\mapsto f(x).$ As maintained by the library, these functions have a few remarkable properties.

*   The function $f$ is periodic, with positive integer period $K\in{\mathbb{N}}+1$ such that $\forall x\in{\mathbb{R}}:f(x)=f(x+K).$
*   The function $f$ is a uniform polynomial spline of nonnegative degree $n\in{\mathbb{N}}$ and shift $\delta x\in{\mathbb{R}},$ which we write as

$$f(x)=\sum_{k\in{\mathbb{Z}}}\,c[{k\bmod K}]\,\beta^{n}(x-\delta x -k).$$
There, $c$ is an arbitrary sequence of coefficients and $\beta^{n}$ is a polynomial B-spline. (B-splines have a finite support and the infinite sum is necessarily well-behaved.) Because the function $f$ is a weighted sum of B-splines shifted by $\delta x+k,$ it inherits from these B-splines important properties such as $n$-times differentiablility and $\left(n-1\right)$-times continuous differentiablility. Moreover, because B-splines have the highest order of approximation for their support, the function $f$ is guaranteed to offer a faithful (in a mathematically precise sense) and computationally efficient representation of data. Finally, the determination of $f(x)$ at any $x\in{\mathbb{R}}$ is computationally exact.

## Defining Parameters
The functions we consider are characterized by parameters that can be chosen freely and independently.

*   Period $K$
*   Degree $n$
*   Delay $\delta x$
*   Spline coefficients $c$

A large variety of functions can be synthesized by the tuning of these parameters, in particular, by the tuning of the spline coefficients. However, it rarely occurs in practice that these coefficients be known beforehand. It is much more common that one is given a vector $\left(s[k]\right)_{k=0}^{K-1}$ of samples, and that it is desired that the synthesized spline interpolates the samples, a requirement that we write as

$$\left(f(k)\right)_{k=0}^{K-1}=\left(s[k]\right)_{k=0}^{K-1}.$$

## Regularizations
Alternatively, it is also sometimes desirable that

$$\left(f(k)\right)_{k=0}^{K-1}\approx\left(s[k]\right)_{k=0}^{K-1},$$
with the approximation being such that it balances the requirement of interpolation with some *a priori* requirement on the continuously defined $f.$ The ``splinekit`` library offers two mechanisms that result in (desirable) approximate interpolation.

*   The first mechanism acknowleges the fact that exact interpolation cannot be achieved (for generic data) in the very specific case of an even period $K\in2\,{\mathbb{N}}+2$ combined with a half-integer delay $\delta x\in{\mathbb{Z}}+1/2.$ In this edge case, stability is recovered by the addition of a constant corrective term to all even samples and by the subtraction of this term from all odd samples. The corrective term is chosen so that the residual difference between $\left(f(k)\right)_{k=0}^{K-1}$ and $\left(s[k]\right)_{k=0}^{K-1}$ is minimized in a least-squares sense under the constraint that $f$ is well-defined. This mechanism is activated by setting ``regularized = True`` in the ``PeriodicSpline1D.from_samples`` creator. It can be thought as a doctoring of the data that would enforce that the component at the highest discrete frequency of their discrete Fourier transform vanishes.
*   The second mechanism is made available in the ``PeriodicSpline1D.from_smoothed_samples`` creator. It acknowledges the fact that the samples may exhibit a local variability that, sometimes, one is inclined to attribute to noise. In this case, the user can choose to restore smoothness to $f$ beyond the smoothness inherited by the B-splines, at some cost in the exactitude of interpolation. More precisely, if we let $\left(\lambda[m]\right)_{m=0}^{n}$ be a vector that represents the weight of variational regularization, then the criterion being minimized is

$$J=\sum_{k=0}^{K-1}\,\left(f(k)-s[k]\right)^{2}+\sum_{m=0}^{n}\,\lambda[m]\,\int_{0}^{K}\,\left(\frac{{\mathrm{d}}^{m}f(x)}{{\mathrm{d}}x^{m}}\right)^{2}\,{\mathrm{d}}x.$$

## From Noiseless Samples
We now propose a few lines of code that create and display a spline that interpolates random samples. The thin gray curve gives the traditional linear interpolation, while the thicker blue curve gives the piecewise polynomial spline. The knots of the spline (the places where the pieces of polynomials meet) are shown as black dots and the spline values at the integers are shown as small circles. The interpolating spline is always well-defined when the period is odd; it is also also always well-defined when the delay is not a half integer. However, exact interpolation is not possible when the period is even and the delay becomes a half integer; this instability can be tamed by regularizing the spline, at the cost of an inexact interpolation.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_samples = 15 # Maximal support of the data samples
max_degree = 5 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay

# Persistent parameters
k0 = -1
n0 = -1
s0 = np.zeros(0)
r0 = sk.interval.Empty()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    regularized = False
):
    global k0 # Period
    global n0 # Degree
    global s0 # Samples
    global r0 # Display range

    # Random samples
    if period != k0 or degree != n0: # Check for the need of new data
        k0 = period
        n0 = degree
        s0 = rng.standard_normal(period) # Fresh samples
        # Fix a display range
        fn = sk.PeriodicSpline1D.from_samples(s0, degree = n0, delay = delay)
        image = fn.image()
        r0 = sk.interval.Closed((
            min(min(s0), image.infimum) - 0.05,
            max(max(s0), image.supremum) + 0.05
        ))
    # Spline of degree n0 from s0 with current delay and regularization
    fn = sk.PeriodicSpline1D.from_samples(
        s0,
        degree = n0,
        delay = delay,
        regularized = regularized
    )
    # Undelayed linear interpolation of the samples
    f1 = sk.PeriodicSpline1D.from_samples(s0, degree = 1)
    # Plot canvas
    (fig, ax) = plt.subplots()
    # Unadorned linear interpolation in thin gray
    f1.plot(
        (fig, ax),
        plotpoints = 301,
        plotrange = r0,
        curve_fmt = "-C7",
        curve_lw = 0.5,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    # Spline in default style
    fn.plot((fig, ax), plotpoints = 301)
    plt.show()

# widgets.interactive(
widgets.interactive(
    update_plot,
    period = (1, max_samples),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    regularized = widgets.Checkbox(value = False, description = 'Regularized')
)


## From Noisy Samples
We now propose a few lines of code that create and display a smooth spline that performs the approximate interpolation of random samples. The thin gray curve gives the traditional linear interpolation, while the thicker blue curve gives the piecewise polynomial spline. The knots of the spline (the places where the pieces of polynomials meet) are shown as black dots and the spline values at the integers are shown as small circles. While a different smoothing weight $\lambda$ could be chosen independently for each order of derivation of the spline, here, for simplicity, we set $\lambda[m]=0$ for $m\in[0\ldots n-1]$ and allow only for the free specification of $\lambda[n],$ where $n$ is the degree of the spline.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_samples = 15 # Maximal support of the data samples
max_degree = 5 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_variational_regularization = 1.2 # Maximal regularization weight

# Persistent parameters
k0 = -1
n0 = -1
s0 = np.zeros(0)
r0 = sk.interval.Empty()

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    smoothing = 0.5
):
    global k0 # Period
    global n0 # Degree
    global s0 # Samples
    global r0 # Display range

    # Random samples
    if period != k0 or degree != n0: # Check for the need of new data
        k0 = period
        n0 = degree
        s0 = rng.standard_normal(period) # Fresh samples
        # Fix a display range
        fn = sk.PeriodicSpline1D.from_samples(s0, degree = n0, delay = delay)
        image = fn.image()
        r0 = sk.interval.Closed((
            min(min(s0), image.infimum) - 0.05,
            max(max(s0), image.supremum) + 0.05
        ))
    # Vector of weights
    lmbd = np.append(np.zeros(n0), smoothing)
    # Spline of degree n0 from s0 with current delay and variational regularization
    fn = sk.PeriodicSpline1D.from_smoothed_samples(
        s0,
        degree = n0,
        delay = delay,
        smoothing = lmbd
    )
    # Undelayed linear interpolation of the samples
    f1 = sk.PeriodicSpline1D.from_samples(s0, degree = 1)
    # Plot canvas
    (fig, ax) = plt.subplots()
    # Unadorned linear interpolation in thin gray
    f1.plot(
        (fig, ax),
        plotpoints = 301,
        plotrange = r0,
        curve_fmt = "-C7",
        curve_lw = 0.5,
        curve_markerfmt = " ",
        curvestem_linefmt = "None",
        knot_marker = " ",
        periodbound_markerfmt = " ",
        periodboundstem_linefmt = "None"
    )
    # Spline in default style
    fn.plot((fig, ax), plotpoints = 301)
    plt.show()

# widgets.interactive(
widgets.interactive(
    update_plot,
    period = (1, max_samples),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    smoothing = (0, max_variational_regularization)
)
